# Runner de experimentos de estabilización (LMI-Net)

Notebook general para **entrenar cualquier arquitectura y medir su rendimiento**
estabilizando sistemas politópicos — sin tener que pedir cada experimento a mano.

- **Arquitecturas**: `vanilla` (baseline, N y n_u fijos), `vertices` (invariante a N),
  `actuadores` (invariante a N y al nº de actuadores n_u).
- **Pérdidas**: elegibles por nombre (`CONFIG["loss"]`) desde un registro extensible
  (sección 3b) — añade pérdidas nuevas sin tocar `benchmark.py`.
- **Métricas comparables**: % estabilizado, loss, tiempo de entrenamiento, memoria
  pico, tiempo de inferencia por sistema, y comparación directa contra **CVXPY**
  (tiempo, factibilidad, y el valor de la pérdida logrado por cada uno — CVXPY solo
  garantiza factibilidad, no que su solución sea óptima bajo ningún criterio).
- **Sección 4**: mete TUS propios sistemas a mano (nominal o por vértices) y
  desplaza sus polos (*pole-shift*) — para los benchmarks del profesor.

Todo el motor está en `analisis/benchmark.py` (así el notebook queda legible).

In [3]:

import warnings; warnings.filterwarnings("ignore")
import numpy as np, torch, pandas as pd
import matplotlib.pyplot as plt
from analisis import benchmark as bm
torch.set_num_threads(6)
print("Listo. Arquitecturas disponibles:", list(bm.ARCHS))

Listo. Arquitecturas disponibles: ['vanilla', 'vertices', 'actuadores']


## 1. Configura tu experimento

Cambia estos hiperparámetros y ejecuta la sección 2.

In [ ]:
CONFIG = dict(
    arch      = "actuadores",  # "vanilla" | "vertices" | "actuadores"
    n         = 3,             # orden del sistema n_x (una arquitectura por orden)
    m         = 1,             # nº de actuadores n_u. Puede ser LISTA, p.ej. [1, 2], para
                               #   entrenar 'actuadores' sobre varios n_u a la vez (evalúa por m y N)
    N_list    = [2, 3, 4],  # nº de vértices a cubrir (invariante en vertices/actuadores)

    dr_train  = 30,            # iteraciones de DR en ENTRENAMIENTO (solo unrolling)
    dr_eval   = 100,          # iteraciones de DR en INFERENCIA / evaluación
    backprop  = "unrolling",   # "unrolling" | "implicit"  (implicit = memoria O(1), gradiente exacto)

    loss      = "control",          # None = default por arquitectura ('paper' en vanilla, 'control' en el
                               #   resto). O el NOMBRE de una pérdida registrada:
                               #   "control" | "paper" | la que tú añadas (ver sección 3b, sin tocar
                               #   benchmark.py: entrenamiento.training.register_loss("nombre", factory))

    epochs    = 100,
    batch     = 16,
    lr        = 1e-3,
    alpha     = 0.001,          # decay-rate objetivo de la LMI (margen de estabilidad)
    limit     = 150,           # sistemas por (orden, N). None = los 500 de la base
    seed      = 42,

    save_failures = True,      # guarda los NO estabilizados en un directorio por combinación:
                               #   resultados/benchmark/<arch>/<backprop>/n{n}_m{m}/
    log_every = 5,             # imprime progreso del entrenamiento cada N épocas (0 = silencioso)
)
CONFIG

{'arch': 'actuadores',
 'n': 3,
 'm': 1,
 'N_list': [2, 3, 4],
 'dr_train': 30,
 'dr_eval': 100,
 'backprop': 'unrolling',
 'loss': 'control',
 'epochs': 100,
 'batch': 16,
 'lr': 0.001,
 'alpha': 0.01,
 'limit': 150,
 'seed': 42,
 'save_failures': True,
 'log_every': 5}

## 2. Corre el experimento

Entrena + evalúa y reporta todas las métricas y gráficos automáticos. La tabla y
las gráficas incluyen `cvxpy_pct`/`cvxpy_ms_per_sys`: % factible y tiempo de
solve de CVXPY (verdad de terreno), para comparar directamente contra el
`stable_pct`/`infer_ms_per_sys` de la red. La pérdida efectivamente usada queda en
`res["config"]["loss"]` (ver `CONFIG["loss"]` arriba y la sección 3b para
comparar pérdidas entre sí).

**Ojo con lo que SÍ y NO significa "CVXPY factible":** CVXPY resuelve solo un
problema de **factibilidad** (el certificado (Q,Y) más cercano a y=0 que cumple
la LMI), **no optimiza ningún criterio de desempeño**. Que sea factible no
implica que sea bueno — p.ej. puede estabilizar con un elipsoide invariante
enorme. La red, en cambio, sí está entrenada para minimizar/maximizar `loss`.
Por eso, además de `cvxpy_pct`, `res["loss_vs_cvxpy"]` trae la pérdida lograda
por cada uno **sobre los mismos sistemas**, y `bm.plot_loss_vs_cvxpy(res)` la
grafica (¿la red encuentra soluciones mejores que "una cualquiera factible"?).

In [13]:
res = bm.run_experiment(**CONFIG)
display(res["df"].round(2))
# si save_failures=True, los sistemas no estabilizados de ESTA combinación quedan en:
if "failures_dir" in res:
    print(f"{res['n_failures']} sistemas NO estabilizados guardados en {res['failures_dir']}")
bm.plot_experiment(res);

# Pérdida de la red vs CVXPY, sistema por sistema (requiere compare_cvxpy=True, default):
if res["pct_red_mejor_loss"] is not None:
    print(f"La red es mejor que CVXPY (según '{res['config']['loss']}', "
          f"{res['config']['loss_direction']}) en el {res['pct_red_mejor_loss']:.1f}% de los sistemas")
    bm.plot_loss_vs_cvxpy(res);

Cargando base de datos desde DB_ssf_RS_500_c.mat...
Dataset inicializado: 500 sistemas válidos encontrados para (n=3, m=1, N=2).
Cargando base de datos desde DB_ssf_RS_500_c.mat...
Dataset inicializado: 500 sistemas válidos encontrados para (n=3, m=1, N=3).
Cargando base de datos desde DB_ssf_RS_500_c.mat...
Dataset inicializado: 500 sistemas válidos encontrados para (n=3, m=1, N=4).
    época    1/100  loss=0.0780  (0.3s/época)
    época    5/100  loss=0.0070  (0.2s/época)
    época   10/100  loss=0.0051  (0.2s/época)
    época   15/100  loss=0.0046  (0.3s/época)
    época   20/100  loss=0.0022  (0.2s/época)
    época   25/100  loss=0.0030  (0.2s/época)
    época   30/100  loss=0.0047  (0.2s/época)
    época   35/100  loss=0.0017  (0.2s/época)
    época   40/100  loss=0.0017  (0.2s/época)
    época   45/100  loss=0.0016  (0.2s/época)
    época   50/100  loss=0.0026  (0.2s/época)
    época   55/100  loss=0.0024  (0.2s/época)
    época   60/100  loss=0.0016  (0.2s/época)
    época   65/

,m,N,stable_pct,decay_pct,worst_mean,infer_ms_per_sys,n,cvxpy_pct,cvxpy_ms_per_sys
0,1,2,46.67,46.67,12.21,0.54,30,95.0,7.71
1,1,3,36.67,33.33,0.36,0.67,30,95.0,104.51
2,1,4,50.00,36.67,0.38,0.91,30,90.0,19.49


50 sistemas NO estabilizados guardados en /Users/pedro/Desktop/Tesis/Red/analisis/resultados/benchmark/actuadores/unrolling/n3_m1
La red es mejor que CVXPY (según 'control', min) en el 1.8% de los sistemas


## 3. Compara arquitecturas × método de backprop

Dos ejes ortogonales:
- **arquitectura**: `vanilla` / `vertices` / `actuadores`
- **backprop**: `unrolling` (gradiente por las iteraciones, memoria O(iters)) vs
  `implicit` (diferenciación implícita, memoria O(1), gradiente exacto)

`compare_grid` corre TODAS las combinaciones y las pone lado a lado
(% estabilizado, tiempo de train, **memoria pico**, y un panel de **inferencia
red vs CVXPY en escala log** para comparar tiempos directamente).

In [ ]:
# Grilla arquitectura × backprop (usa dr_train alto para que se note la memoria)
tabla, results, fig = bm.compare_grid(
    archs=["vertices", "actuadores"],
    backprops=["unrolling", "implicit"],
    n=3, N_list=[2,3,4,5], m=1, dr_train=30, dr_eval=1000, epochs=15, limit=150)
display(tabla.round(2))

# Comparaciones sueltas también sirven, p. ej. el efecto del truncamiento:
# rA = bm.run_experiment(arch="actuadores", backprop="unrolling", dr_train=30,  epochs=15)
# rB = bm.run_experiment(arch="actuadores", backprop="unrolling", dr_train=500, epochs=15)
# bm.compare([rA, rB], labels=["unroll-30", "unroll-500"]);

## 3b. Grid search sobre pérdidas (y cualquier otro hiperparámetro)

Las pérdidas están en un **registro por nombre** (`entrenamiento.training.LOSS_REGISTRY`):
hoy `"control"` (volumen + esfuerzo) y `"paper"` (Ec. 21 del paper). Para añadir una
pérdida nueva **sin tocar `benchmark.py`**:

```python
from entrenamiento import training as T

def mi_perdida_factory(model):
    # recibe el modelo (por si necesitas model.alpha, model.epsilon, ...)
    return lambda Q, Y, A_poly, B_poly: ...  # tu pérdida, mismo formato (Q,Y,A,B)->escalar

T.register_loss("mi_perdida", mi_perdida_factory)
```

y ya queda disponible como `loss="mi_perdida"` en `run_experiment`, `compare_grid` y
`grid_search`.

`bm.grid_search(param_grid, **fijos)` corre **todas las combinaciones** de
`param_grid` (cualquier parámetro de `run_experiment`: `loss`, `backprop`, `dr_train`,
`alpha`, `arch`, ...) y devuelve una tabla comparativa + la lista de resultados
completos (para graficar con `plot_experiment` cualquiera de ellos).

In [ ]:
from entrenamiento import training as T
print("Pérdidas registradas:", list(T.LOSS_REGISTRY))

# Grid search: pérdida × backprop, arquitectura fija (actuadores, n_x=3, N=2).
# compare_cvxpy=True (con cvxpy_max_systems chico) para ademas comparar, por cada
# combinacion, si la red logra mejor 'loss' que el certificado factible de CVXPY.
tabla_gs, results_gs = bm.grid_search(
    {"loss": ["control", "paper"], "backprop": ["unrolling", "implicit"]},
    arch="actuadores", n=3, N_list=[2], m=1, dr_train=30, dr_eval=1000,
    epochs=15, limit=150, compare_cvxpy=True, cvxpy_max_systems=15)
display(tabla_gs.round(3))
bm.plot_grid_search(tabla_gs, x="loss", hue="backprop", y="stable_pct")
bm.plot_grid_search(tabla_gs, x="loss", hue="backprop", y="pct_red_mejor_loss",
                    title="% de sistemas donde la red supera a CVXPY en la pérdida");

# Inspeccionar la comparación sistema-por-sistema de una combinación puntual del grid:
# bm.plot_loss_vs_cvxpy(results_gs[0])

# Otro barrido típico: solo pérdida, con más épocas (deja backprop y arch fijos)
# tabla_gs2, _ = bm.grid_search({"loss": list(T.LOSS_REGISTRY)},
#                                arch="vertices", n=3, N_list=[2,3,4], m=1, epochs=15)
# bm.plot_grid_search(tabla_gs2, x="loss", y="stable_pct");

### Baseline: vanilla fiel al paper LMI-Net

`arch="vanilla"` reproduce la receta del apéndice del paper: MLP 64×64 ReLU,
**backward implícito**, **500 iteraciones DR fijas**, `epsilon=1e-3` y la **pérdida
del paper** (Ec. 21) — `run_experiment` la aplica automáticamente para vanilla
(verás `config['loss']=='paper'`). La vanilla es de N fijo (una arquitectura por N);
la celda de reproducción del paper es n_x=3, N=2.

Reproducción completa (1000 épocas, tabla por iteraciones DR):
`analisis/experimento_vanilla_paper.py`.

In [ ]:
# Baseline vanilla FIEL al paper. El implícito con 500 iters DR es LENTO
# (~0.3s/época con limit=150); log_every muestra el avance. Súbelo a 1000 épocas
# para el paper (o usa el script dedicado).
res_vanilla = bm.run_experiment(arch="vanilla", n=3, N_list=[2], m=1,
                                backprop="implicit", dr_train=500, dr_eval=2000,
                                epochs=100, limit=CONFIG["limit"], log_every=10)
display(res_vanilla["df"].round(3))
print("pérdida usada:", res_vanilla["config"]["loss"])   # -> 'paper'
bm.plot_experiment(res_vanilla);

# Comparar la baseline vanilla (paper) contra una contribución (misma celda n_x=3, N=2):
# res_act = bm.run_experiment(arch="actuadores", n=3, N_list=[2], m=1, epochs=CONFIG["epochs"])
# bm.compare([res_vanilla, res_act], labels=["vanilla (paper)", "actuadores"]);

## 4. Probar sistemas propios

Usa un modelo de la sección 2/3 que sea **invariante**
(`vertices` o `actuadores`). Hay dos formas de meter sistemas:

**(a) Sistema NOMINAL** (un solo par (A, B), no politopo):
```python
A = [[0,1,0],[0,0,1],[2,-3,1]]   # matriz de estado n×n
B = [[0],[0],[1]]                # matriz de entrada n×m  (m columnas = m actuadores)
sistema = bm.system_to_polytope(A, B)     # -> politopo con N=1
```

**(b) POLITOPO por vértices** (varios (A_i, B_i)):
```python
sistema = bm.polytope_from_vertices([A1, A2], [B1, B2])   # N=2 vértices
```

**Pole-shift (desplazar los polos un valor `shift`):** `bm.shift_poles(A_poly, shift)`
hace `A → A + shift·I`, es decir suma `shift` a la parte real de TODOS los polos.
`shift>0` vuelve el sistema más inestable; `shift<0`, más estable. (OJO: `shift` NO
es el `alpha` de la LMI; son cosas distintas.) Para un solo desplazamiento, p.ej.
de 1: `Ap = bm.shift_poles(Ap, 1.0)`. Para barrer varios, usa un array de `shift`.

**Antes de correr nada**, conviene chequear si el sistema es siquiera estabilizable
con el `alpha` del modelo:
```python
bm.check_stabilizable(model, A_poly, B_poly)   # resuelve CVXPY, imprime SI/NO
```
Si es infactible, ninguna cantidad de iteraciones de DR (ni de entrenamiento) lo
va a estabilizar — es una propiedad del sistema/alpha, no un problema del solver.

**¿Cuántas iteraciones le cuesta a la red llegar a una solución?**
```python
bm.iters_to_stabilize(model, A_poly, B_poly, budgets=[100,500,1000,2000,4000,8000])
```
Devuelve el **menor** presupuesto de iteraciones DR que ya estabiliza el politopo
(o `None` si no lo logra con ninguno de los probados), más el peor autovalor a
cada presupuesto — para ver el sistema acercándose a la estabilidad iteración a
iteración. Es barato: una sola pasada de DR con checkpoints, no una corrida por
presupuesto.

Finalmente, evalúa las demás métricas con:
```python
bm.benchmark_systems(model, [sistema1, sistema2, ...], dr_eval=1000)
```
- Cada `sistema` es la tupla `(A_poly, B_poly)` que devuelven los helpers de arriba.
- Por defecto **normaliza** cada sistema igual que en el entrenamiento
  (`normalize=True`).
- **Siempre** compara contra CVXPY (`compare_cvxpy=True` por defecto): añade la
  columna `lmi_factible_cvxpy` — verdad de terreno, existe certificado con margen
  α — que distingue "el algoritmo falló" de "no hay solución con ese α". Pásale
  `compare_cvxpy=False` solo si necesitas saltarte el solver SDP por velocidad.
- Pásale `loss="control"|"paper"|...` para además comparar, en esos mismos
  sistemas, el valor de esa pérdida logrado por la red (`loss_red`) contra el
  de CVXPY (`loss_cvxpy`) — recuerda que CVXPY solo garantiza factibilidad, no
  que su solución sea buena bajo ese criterio (ver sección 2).

Esta misma comparación contra CVXPY también sale sola en las secciones 2 y 3
(`run_experiment`/`compare_grid` añaden `cvxpy_pct` al df y a las gráficas de
`compare`/`compare_grid`), para que cualquier benchmark de este notebook incluya
la verdad de terreno junto al % estabilizado por la red.

In [14]:
# --- 1) modelo invariante ya entrenado (de la sección 2) ---
model = res["model"]

# --- 2) EDITA aquí tu sistema (nominal) ---
A = [[0, 1, 0],
     [0, 0, 1],
     [2,-3, 1]]            # n×n
B = [[0],
     [0],
     [1]]                  # n×m

# Presupuestos de iteraciones DR a probar para medir "cuánto le cuesta llegar a
# una solución" (iters_to_stabilize busca el menor de estos que ya estabiliza).
BUDGETS = [100, 250, 500, 1000, 2000, 4000, 8000]

# --- 3) Barre el desplazamiento de polos hacia alpha ---
shifts = np.linspace(-3.0, 2.0, 10)
rows = []
for s in shifts:
    Ap, Bp = bm.system_to_polytope(A, B)     # nominal -> politopo (N=1)
    Ap = bm.shift_poles(Ap, s)               # A -> A + s*I

    # ANTES de correr nada: ¿existe siquiera un certificado factible con este alpha?
    # Si es infactible, ninguna cantidad de iteraciones lo va a estabilizar.
    factible = bm.check_stabilizable(model, Ap, Bp)

    # loss=CONFIG["loss"] o cualquier otra registrada: compara, en CADA punto del
    # barrido, la pérdida lograda por la red vs por CVXPY (certificado factible).
    df = bm.benchmark_systems(model, [(Ap, Bp)], dr_eval=CONFIG["dr_eval"],
                              loss=CONFIG["loss"] or "control")
    r = df.iloc[0].to_dict(); r["shift"] = s; r["lmi_factible"] = factible

    # ¿Cuántas iteraciones DR necesita la red para estabilizar ESTE politopo?
    it_info = bm.iters_to_stabilize(model, Ap, Bp, budgets=BUDGETS)
    r["iters_min"] = it_info["iters_min"]           # None = no logrado con estos budgets
    r["estabilizado_en_max_iters"] = it_info["estabilizado_en_max"]

    rows.append(r)

sweep = pd.DataFrame(rows)
cols = ["shift", "lmi_factible", "estabilizado", "iters_min", "decay_logrado",
       "peor_autovalor_cl", "infer_ms", "cvxpy_ms"]
if "loss_red" in sweep:
    cols += ["loss_red", "loss_cvxpy"]
display(sweep[cols].round(3))
bm.plot_pole_shift(sweep);

# Costo en iteraciones para estabilizar, vs shift (escala log). Los que NO
# estabilizaron ni con el mayor budget se grafican por encima de la línea punteada.
fig, ax = plt.subplots(figsize=(6.4, 3.6))
no_logrado = sweep.iters_min.isna()
ax.plot(sweep.loc[~no_logrado, "shift"], sweep.loc[~no_logrado, "iters_min"],
       marker="o", color="#2471a3", label="iteraciones necesarias")
if no_logrado.any():
    ax.scatter(sweep.loc[no_logrado, "shift"], [max(BUDGETS)] * no_logrado.sum(),
              marker="x", color="#a93226", s=60, label=f"no logrado ni con {max(BUDGETS)}")
ax.axhline(max(BUDGETS), ls="--", color="k", alpha=.4, lw=1)
ax.set_yscale("log")
ax.set(xlabel="shift aplicado a los polos", ylabel="iteraciones DR necesarias (log)",
      title="Costo de iteraciones para estabilizar vs desplazamiento de polos")
ax.legend(fontsize=8); fig.tight_layout()

[chequeo previo] N=1 alpha=0.01 epsilon=1e-05: SI existe certificado factible -> con iteraciones suficientes, la red DEBERIA estabilizarlo  (13 ms CVXPY)
[chequeo previo] N=1 alpha=0.01 epsilon=1e-05: SI existe certificado factible -> con iteraciones suficientes, la red DEBERIA estabilizarlo  (4 ms CVXPY)
[chequeo previo] N=1 alpha=0.01 epsilon=1e-05: SI existe certificado factible -> con iteraciones suficientes, la red DEBERIA estabilizarlo  (4 ms CVXPY)
[chequeo previo] N=1 alpha=0.01 epsilon=1e-05: SI existe certificado factible -> con iteraciones suficientes, la red DEBERIA estabilizarlo  (4 ms CVXPY)
[chequeo previo] N=1 alpha=0.01 epsilon=1e-05: SI existe certificado factible -> con iteraciones suficientes, la red DEBERIA estabilizarlo  (4 ms CVXPY)
[chequeo previo] N=1 alpha=0.01 epsilon=1e-05: SI existe certificado factible -> con iteraciones suficientes, la red DEBERIA estabilizarlo  (5 ms CVXPY)
[chequeo previo] N=1 alpha=0.01 epsilon=1e-05: SI existe certificado factible -> 

,shift,lmi_factible,estabilizado,iters_min,decay_logrado,peor_autovalor_cl,infer_ms,cvxpy_ms,loss_red,loss_cvxpy
0,-3.000,True,True,100.0,0.966,-0.966,7.372,6.110,0.001,0.000
1,-2.444,True,True,100.0,0.765,-0.765,3.975,3.601,0.001,0.000
2,-1.889,True,True,100.0,0.556,-0.556,3.841,3.760,0.002,0.000
3,-1.333,True,True,100.0,0.355,-0.355,4.248,4.246,0.001,0.000
4,-0.778,True,True,100.0,0.170,-0.170,3.711,3.900,0.001,0.000
5,-0.222,True,False,250.0,-0.092,0.092,3.802,4.026,0.000,0.000
6,0.333,True,False,NaN,-0.249,0.249,3.810,4.575,0.000,0.000
7,0.889,True,False,NaN,-1.197,1.197,3.763,4.442,-0.000,0.000
8,1.444,True,False,NaN,-16.735,16.735,3.677,4.827,0.000,0.001
9,2.000,True,False,NaN,-47.609,47.609,3.703,6.672,0.000,0.004


In [14]:
sweep.to_csv("poleshift_sweep_loss_paper.csv", index=False)

### 4c. Sistema del profesor (pizarra): politopo con desplazamiento POR VÉRTICE

Lo de la pizarra es un **politopo de N=2 vértices, orden n_x=2**, donde cada
vértice se perturba con su **propia matriz diagonal** (no un `shift·I` uniforme):

$$A_1(\delta)=\begin{bmatrix}0&1\\-2&-2\end{bmatrix}+\delta\begin{bmatrix}2&0\\0&1\end{bmatrix},\quad
A_2(\delta)=\begin{bmatrix}0&1\\-2&-3\end{bmatrix}+\delta\begin{bmatrix}-2&0\\0&1\end{bmatrix},\quad
B=\begin{bmatrix}0\\1\end{bmatrix}\ \text{(genérica, igual en ambos vértices)}$$

Al crecer $\delta$ los polos se corren a la derecha pero **a ritmos distintos por
vértice** (el vértice 2 se desestabiliza más rápido por el 5). Soportado con
`bm.shift_poles(A_poly, delta, directions=[D1, D2])`, y `polytope_from_vertices`
acepta una sola `B` genérica que se replica a todos los vértices.

**OJO — orden**: este sistema es $n_x=2$, y el modelo de la sección 2 es de orden 3.
Las arquitecturas son invariantes a N (y `actuadores` a $n_u$) pero **no al orden**,
así que la celda entrena un modelo con `n=2` (la base tiene $n_x=2$ solo para $n_u=1$).

In [ ]:
# === Sistema del profesor: A_i(delta) = A_i0 + delta*D_i,  B generica ===
A1 = [[0, 1], [-2, -2]];   D1 = [[2, 0], [0, 1]]
A2 = [[0, 1], [-2, -3]];   D2 = [[-2, 0], [0, 1]]
B  = [[0], [1]]

BUDGETS = [100, 250, 500, 1000, 2000, 4000, 8000]

# El sistema es de ORDEN 2 -> modelo propio (el de la sección 2 es n=3 y daría error claro).
res_n2 = bm.run_experiment(arch="actuadores", n=2, N_list=[2, 3], m=1,
                           dr_train=CONFIG["dr_train"], dr_eval=CONFIG["dr_eval"],
                           backprop=CONFIG["backprop"], loss=CONFIG["loss"],
                           epochs=CONFIG["epochs"], limit=CONFIG["limit"],
                           compare_cvxpy=False, save_failures=False,
                           log_every=CONFIG["log_every"])
model_n2 = res_n2["model"]

# Politopo base (delta=0). B unica -> se replica a los 2 vertices.
Ap0, Bp0 = bm.polytope_from_vertices([A1, A2], [B])

# OJO: benchmark_systems evalua a dr_eval=CONFIG["dr_eval"] iteraciones. Si es bajo
# (p.ej. 100), 'estabilizado' saldra False para deltas que SI se logran con mas
# iteraciones — la columna que manda es iters_min / peor_eig_max_iters.
DR_EVAL_SISTEMA = max(CONFIG["dr_eval"], 2000)

deltas = np.linspace(0.0, 1.5, 11)
rows = []
for d in deltas:
    Ap = bm.shift_poles(Ap0, d, directions=[D1, D2])   # A_i -> A_i + d*D_i
    factible = bm.check_stabilizable(model_n2, Ap, Bp0, verbose=False)
    df = bm.benchmark_systems(model_n2, [(Ap, Bp0)], dr_eval=DR_EVAL_SISTEMA,
                              loss=CONFIG["loss"] or "control")
    r = df.iloc[0].to_dict(); r["delta"] = d; r["lmi_factible"] = factible
    # autovalores de LAZO ABIERTO de cada vertice en este delta (para el CSV)
    for i in range(len(Ap)):
        ev = np.linalg.eigvals(Ap[i])
        r[f"eig_A{i+1}"] = " ".join(f"{e:.3f}" for e in np.round(ev, 3))
        r[f"absc_A{i+1}"] = float(ev.real.max())
    it_info = bm.iters_to_stabilize(model_n2, Ap, Bp0, budgets=BUDGETS)
    r["iters_min"] = it_info["iters_min"]
    r["peor_eig_max_iters"] = it_info["worst_eig_by_iters"][max(BUDGETS)]
    rows.append(r)

sweep_prof = pd.DataFrame(rows)
cols = ["delta", "eig_A1", "eig_A2", "lmi_factible", "estabilizado", "iters_min",
       "peor_eig_max_iters", "decay_logrado", "infer_ms", "cvxpy_ms"]
if "loss_red" in sweep_prof:
    cols += ["loss_red", "loss_cvxpy"]
display(sweep_prof[cols].round(3))
sweep_prof.to_csv("poleshift_profesor.csv", index=False)

fig, axs = plt.subplots(1, 3, figsize=(14, 3.6))
axs[0].plot(sweep_prof.delta, sweep_prof.absc_A1, marker="o", label="vértice 1", color="#a93226")
axs[0].plot(sweep_prof.delta, sweep_prof.absc_A2, marker="s", label="vértice 2", color="#2471a3")
axs[0].axhline(0, ls="--", color="k", alpha=.5)
axs[0].set(xlabel=r"$\delta$", ylabel=r"$\max\,\mathrm{Re}\,\lambda$ (lazo abierto)",
          title="Abscisa de cada vértice vs $\\delta$")
axs[0].legend(fontsize=8)
axs[1].plot(sweep_prof.delta, sweep_prof.peor_eig_max_iters, marker="o", color="#2471a3")
axs[1].axhline(0, ls="--", color="k", alpha=.5)
axs[1].set(xlabel=r"$\delta$", ylabel=f"peor autovalor CL @ {max(BUDGETS)} iters",
          title="Lazo cerrado al máximo presupuesto")
ok = sweep_prof.iters_min.notna()
axs[2].plot(sweep_prof.loc[ok, "delta"], sweep_prof.loc[ok, "iters_min"], marker="o", color="#1e8449")
if (~ok).any():
    axs[2].scatter(sweep_prof.loc[~ok, "delta"], [max(BUDGETS)] * (~ok).sum(),
                  marker="x", color="#a93226", s=60, label=f"no logrado ni con {max(BUDGETS)}")
    axs[2].legend(fontsize=8)
axs[2].set_yscale("log")
axs[2].set(xlabel=r"$\delta$", ylabel="iteraciones DR necesarias (log)",
          title="Costo de iteraciones vs $\\delta$")
fig.tight_layout()